# Evoformer Explained: The Heart of AlphaFold2

This notebook provides a hands-on exploration of the Evoformer architecture, the core component of AlphaFold2 that processes multiple sequence alignments (MSAs) and pairwise relationships to predict protein structure.

## Learning Objectives

1. Understand MSA and pair representations
2. Implement triangular multiplicative updates
3. Implement triangular self-attention
4. See how information flows between MSA and pair representations

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Understanding the Representations

AlphaFold2 maintains two key representations:

1. **MSA Representation** $m_{si}$: Shape `[N_seq, L, c_m]`
   - For each sequence $s$ and position $i$, stores learned features
   - Captures evolutionary information from homologous sequences

2. **Pair Representation** $z_{ij}$: Shape `[L, L, c_z]`
   - For each pair of positions $(i, j)$, stores relationship features
   - Encodes spatial proximity, co-evolution signals, etc.

In [ ]:
# Define dimensions
L = 50          # Sequence length
N_seq = 128     # Number of sequences in MSA
c_m = 256       # MSA feature dimension
c_z = 128       # Pair feature dimension

# Create example representations
msa_repr = torch.randn(N_seq, L, c_m)
pair_repr = torch.randn(L, L, c_z)

print(f"MSA representation shape: {msa_repr.shape}")
print(f"Pair representation shape: {pair_repr.shape}")
print(f"\nTotal parameters in MSA: {N_seq * L * c_m:,}")
print(f"Total parameters in Pair: {L * L * c_z:,}")

## 2. MSA Row Attention with Pair Bias

The first key operation is attention over positions **within each sequence**, biased by the pair representation.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + b_{ij}\right)V$$

where $b_{ij}$ comes from the pair representation.

In [ ]:
class MSARowAttentionWithPairBias(nn.Module):
    """MSA row-wise attention with pair representation bias."""
    
    def __init__(self, c_m, c_z, n_heads=8):
        super().__init__()
        self.c_m = c_m
        self.n_heads = n_heads
        self.head_dim = c_m // n_heads
        
        self.layer_norm_m = nn.LayerNorm(c_m)
        self.layer_norm_z = nn.LayerNorm(c_z)
        
        # Q, K, V projections
        self.to_q = nn.Linear(c_m, c_m, bias=False)
        self.to_k = nn.Linear(c_m, c_m, bias=False)
        self.to_v = nn.Linear(c_m, c_m, bias=False)
        
        # Pair bias: project pair features to attention bias
        self.pair_bias = nn.Linear(c_z, n_heads, bias=False)
        
        # Output projection
        self.to_out = nn.Linear(c_m, c_m)
        
    def forward(self, msa_repr, pair_repr):
        N_seq, L, _ = msa_repr.shape
        
        # Normalize inputs
        m = self.layer_norm_m(msa_repr)
        z = self.layer_norm_z(pair_repr)
        
        # Compute Q, K, V
        q = self.to_q(m).view(N_seq, L, self.n_heads, self.head_dim)
        k = self.to_k(m).view(N_seq, L, self.n_heads, self.head_dim)
        v = self.to_v(m).view(N_seq, L, self.n_heads, self.head_dim)
        
        # Compute attention scores: [N_seq, n_heads, L, L]
        attn = torch.einsum('bihd,bjhd->bhij', q, k) / (self.head_dim ** 0.5)
        
        # Add pair bias: [L, L, c_z] -> [L, L, n_heads] -> [1, n_heads, L, L]
        bias = self.pair_bias(z).permute(2, 0, 1).unsqueeze(0)
        attn = attn + bias
        
        # Softmax and apply to values
        attn_weights = torch.softmax(attn, dim=-1)
        out = torch.einsum('bhij,bjhd->bihd', attn_weights, v)
        out = out.reshape(N_seq, L, self.c_m)
        
        return msa_repr + self.to_out(out), attn_weights

# Test the module
msa_attn = MSARowAttentionWithPairBias(c_m, c_z)
msa_out, attn_weights = msa_attn(msa_repr, pair_repr)

print(f"Input MSA shape: {msa_repr.shape}")
print(f"Output MSA shape: {msa_out.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

In [ ]:
# Visualize attention pattern for the first sequence, first head
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Attention weights
im1 = axes[0].imshow(attn_weights[0, 0].detach().numpy(), cmap='Blues')
axes[0].set_title('Attention Weights (Seq 0, Head 0)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')
plt.colorbar(im1, ax=axes[0])

# Mean attention across all sequences and heads
mean_attn = attn_weights.mean(dim=(0, 1)).detach().numpy()
im2 = axes[1].imshow(mean_attn, cmap='Blues')
axes[1].set_title('Mean Attention (All Seqs & Heads)')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

## 3. Triangular Multiplicative Update

The pair representation needs to maintain geometric consistency. The triangular update enforces: if residue $i$ is close to $j$, and $j$ is close to $k$, then $i$ should have a consistent relationship with $k$.

### Outgoing Edges Update:
$$z_{ij} \leftarrow z_{ij} + \sum_k a_{ik} \odot b_{jk}$$

This aggregates information along outgoing edges from $i$ and $j$ to common neighbors $k$.

### Incoming Edges Update:
$$z_{ij} \leftarrow z_{ij} + \sum_k a_{ki} \odot b_{kj}$$

This aggregates information along incoming edges from common predecessors $k$.

In [ ]:
class TriangularMultiplicativeUpdate(nn.Module):
    """Triangular multiplicative update for pair representation."""
    
    def __init__(self, c_z, c_hidden=128, mode='outgoing'):
        super().__init__()
        self.c_z = c_z
        self.mode = mode
        
        self.layer_norm = nn.LayerNorm(c_z)
        
        # Projection layers
        self.left_proj = nn.Linear(c_z, c_hidden)
        self.right_proj = nn.Linear(c_z, c_hidden)
        
        # Gates
        self.left_gate = nn.Linear(c_z, c_hidden)
        self.right_gate = nn.Linear(c_z, c_hidden)
        
        # Output
        self.output_gate = nn.Linear(c_z, c_z)
        self.output_proj = nn.Linear(c_hidden, c_z)
        self.final_norm = nn.LayerNorm(c_hidden)
        
    def forward(self, pair_repr):
        z = self.layer_norm(pair_repr)
        
        # Gated projections
        left = self.left_proj(z) * torch.sigmoid(self.left_gate(z))
        right = self.right_proj(z) * torch.sigmoid(self.right_gate(z))
        
        # Triangular aggregation
        if self.mode == 'outgoing':
            # z_ij = sum_k (left_ik * right_jk)
            out = torch.einsum('ikc,jkc->ijc', left, right)
        else:  # incoming
            # z_ij = sum_k (left_ki * right_kj)
            out = torch.einsum('kic,kjc->ijc', left, right)
        
        # Output with gate
        out = self.final_norm(out)
        out = self.output_proj(out)
        gate = torch.sigmoid(self.output_gate(pair_repr))
        
        return pair_repr + gate * out

# Test both modes
tri_out = TriangularMultiplicativeUpdate(c_z, mode='outgoing')
tri_in = TriangularMultiplicativeUpdate(c_z, mode='incoming')

pair_out = tri_out(pair_repr)
pair_in = tri_in(pair_repr)

print(f"Input pair shape: {pair_repr.shape}")
print(f"Output (outgoing) shape: {pair_out.shape}")
print(f"Output (incoming) shape: {pair_in.shape}")

In [ ]:
# Visualize the effect of triangular update
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Show one channel of pair representation
channel = 0

im1 = axes[0].imshow(pair_repr[:, :, channel].detach().numpy(), cmap='RdBu_r', vmin=-2, vmax=2)
axes[0].set_title(f'Original Pair Rep (channel {channel})')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(pair_out[:, :, channel].detach().numpy(), cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_title(f'After Outgoing Update')
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(pair_in[:, :, channel].detach().numpy(), cmap='RdBu_r', vmin=-2, vmax=2)
axes[2].set_title(f'After Incoming Update')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

## 4. Triangular Self-Attention

Triangular attention allows each position pair $(i, j)$ to attend to other pairs that share one endpoint.

### Starting Node Attention:
For row $i$, elements $z_{ij}$ attend to all $z_{ik}$ (same starting node).

### Ending Node Attention:
For column $j$, elements $z_{ij}$ attend to all $z_{kj}$ (same ending node).

In [ ]:
class TriangularSelfAttention(nn.Module):
    """Triangular self-attention for pair representation."""
    
    def __init__(self, c_z, n_heads=4, mode='starting'):
        super().__init__()
        self.c_z = c_z
        self.n_heads = n_heads
        self.head_dim = c_z // n_heads
        self.mode = mode
        
        self.layer_norm = nn.LayerNorm(c_z)
        
        self.to_q = nn.Linear(c_z, c_z, bias=False)
        self.to_k = nn.Linear(c_z, c_z, bias=False)
        self.to_v = nn.Linear(c_z, c_z, bias=False)
        
        # Bias from pair representation itself
        self.bias_proj = nn.Linear(c_z, n_heads, bias=False)
        
        self.to_out = nn.Linear(c_z, c_z)
        self.gate = nn.Linear(c_z, c_z)
        
    def forward(self, pair_repr):
        L = pair_repr.shape[0]
        
        if self.mode == 'ending':
            pair_repr = pair_repr.transpose(0, 1)
        
        z = self.layer_norm(pair_repr)
        
        # QKV projections: [L, L, c_z] -> [L, L, n_heads, head_dim]
        q = self.to_q(z).view(L, L, self.n_heads, self.head_dim)
        k = self.to_k(z).view(L, L, self.n_heads, self.head_dim)
        v = self.to_v(z).view(L, L, self.n_heads, self.head_dim)
        
        # For each row i, compute attention over columns
        # q[i,j,:,:] attends to k[i,k,:,:]
        attn = torch.einsum('ijhd,ikhd->hijk', q, k) / (self.head_dim ** 0.5)
        
        # Add bias from pair representation
        bias = self.bias_proj(z).permute(2, 0, 1)  # [n_heads, L, L]
        attn = attn + bias.unsqueeze(1)  # broadcast over j dimension
        
        attn_weights = torch.softmax(attn, dim=-1)
        out = torch.einsum('hijk,ikhd->ijhd', attn_weights, v)
        out = out.reshape(L, L, self.c_z)
        
        gate = torch.sigmoid(self.gate(pair_repr))
        result = pair_repr + gate * self.to_out(out)
        
        if self.mode == 'ending':
            result = result.transpose(0, 1)
            
        return result, attn_weights

# Test both modes
tri_attn_start = TriangularSelfAttention(c_z, mode='starting')
tri_attn_end = TriangularSelfAttention(c_z, mode='ending')

pair_start, attn_start = tri_attn_start(pair_repr)
pair_end, attn_end = tri_attn_end(pair_repr)

print(f"Starting node attention output: {pair_start.shape}")
print(f"Starting node attention weights: {attn_start.shape}")
print(f"Ending node attention output: {pair_end.shape}")

In [ ]:
# Visualize triangular attention pattern
# For starting node attention: show how position (i, j) attends over k

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Pick a specific row to visualize
row_i = 10
head_h = 0

# Starting node attention: for row i, show attention pattern
# attn_start[h, i, j, k] = how much z_{ij} attends to z_{ik}
start_attn_row = attn_start[head_h, row_i].detach().numpy()

im1 = axes[0, 0].imshow(start_attn_row, cmap='Blues')
axes[0, 0].set_title(f'Starting Node Attn (row={row_i}, head={head_h})')
axes[0, 0].set_xlabel('Key k (attending to z_ik)')
axes[0, 0].set_ylabel('Query j (from z_ij)')
plt.colorbar(im1, ax=axes[0, 0])

# Show different rows
for idx, row in enumerate([5, 25, 40]):
    ax = axes[0, idx] if idx == 0 else axes[1, idx-1]
    if idx > 0:
        im = axes[1, idx-1].imshow(attn_start[head_h, row].detach().numpy(), cmap='Blues')
        axes[1, idx-1].set_title(f'Starting Node Attn (row={row})')
        axes[1, idx-1].set_xlabel('Key k')
        axes[1, idx-1].set_ylabel('Query j')
        plt.colorbar(im, ax=axes[1, idx-1])

# Mean attention pattern
mean_start_attn = attn_start.mean(dim=(0, 1)).detach().numpy()
im = axes[0, 1].imshow(mean_start_attn, cmap='Blues')
axes[0, 1].set_title('Mean Starting Node Attention')
plt.colorbar(im, ax=axes[0, 1])

# Diagonal dominance check
diag_values = np.diag(mean_start_attn)
axes[0, 2].plot(diag_values)
axes[0, 2].set_title('Diagonal Attention Values')
axes[0, 2].set_xlabel('Position')
axes[0, 2].set_ylabel('Self-Attention Weight')

plt.tight_layout()
plt.show()

## 5. Outer Product Mean: MSA to Pair Communication

The outer product mean transfers information from the MSA representation to the pair representation:

$$z_{ij} \leftarrow z_{ij} + \frac{1}{N_{seq}} \sum_s \text{Linear}(m_{si}) \otimes \text{Linear}(m_{sj})$$

This captures co-evolution: if positions $i$ and $j$ co-vary across sequences, their outer product will have high values.

In [ ]:
class OuterProductMean(nn.Module):
    """Communicate MSA information to pair representation."""
    
    def __init__(self, c_m, c_z, c_hidden=32):
        super().__init__()
        self.c_hidden = c_hidden
        
        self.layer_norm = nn.LayerNorm(c_m)
        self.left_proj = nn.Linear(c_m, c_hidden)
        self.right_proj = nn.Linear(c_m, c_hidden)
        self.output = nn.Linear(c_hidden * c_hidden, c_z)
        
    def forward(self, msa_repr):
        N_seq, L, _ = msa_repr.shape
        
        m = self.layer_norm(msa_repr)
        
        # Project to smaller dimension
        left = self.left_proj(m)   # [N_seq, L, c_hidden]
        right = self.right_proj(m) # [N_seq, L, c_hidden]
        
        # Outer product: for each sequence, compute [L, L, c_hidden, c_hidden]
        # Then mean over sequences
        outer = torch.einsum('sic,sjd->ijcd', left, right) / N_seq
        
        # Flatten and project to c_z
        outer = outer.reshape(L, L, self.c_hidden * self.c_hidden)
        return self.output(outer)

# Test
opm = OuterProductMean(c_m, c_z)
pair_update = opm(msa_repr)

print(f"MSA input shape: {msa_repr.shape}")
print(f"Pair update shape: {pair_update.shape}")

In [ ]:
# Visualize the outer product mean output
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Show a few channels
for idx, ch in enumerate([0, 32, 64]):
    im = axes[idx].imshow(pair_update[:, :, ch].detach().numpy(), cmap='RdBu_r')
    axes[idx].set_title(f'Outer Product Mean (channel {ch})')
    axes[idx].set_xlabel('Position j')
    axes[idx].set_ylabel('Position i')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.show()

## 6. Complete Evoformer Block

Now let's put it all together into a complete Evoformer block.

In [ ]:
class EvoformerBlock(nn.Module):
    """Complete Evoformer block."""
    
    def __init__(self, c_m=256, c_z=128, n_heads_msa=8, n_heads_pair=4):
        super().__init__()
        
        # MSA stack
        self.msa_row_attn = MSARowAttentionWithPairBias(c_m, c_z, n_heads_msa)
        self.msa_col_attn = self._build_msa_col_attn(c_m, n_heads_msa)
        self.msa_transition = nn.Sequential(
            nn.LayerNorm(c_m),
            nn.Linear(c_m, c_m * 4),
            nn.ReLU(),
            nn.Linear(c_m * 4, c_m)
        )
        
        # Pair stack
        self.tri_mult_out = TriangularMultiplicativeUpdate(c_z, mode='outgoing')
        self.tri_mult_in = TriangularMultiplicativeUpdate(c_z, mode='incoming')
        self.tri_attn_start = TriangularSelfAttention(c_z, n_heads_pair, mode='starting')
        self.tri_attn_end = TriangularSelfAttention(c_z, n_heads_pair, mode='ending')
        self.pair_transition = nn.Sequential(
            nn.LayerNorm(c_z),
            nn.Linear(c_z, c_z * 4),
            nn.ReLU(),
            nn.Linear(c_z * 4, c_z)
        )
        
        # MSA to Pair communication
        self.outer_product_mean = OuterProductMean(c_m, c_z)
        
    def _build_msa_col_attn(self, c_m, n_heads):
        """Simple column attention (no pair bias)."""
        return nn.MultiheadAttention(c_m, n_heads, batch_first=True)
    
    def forward(self, msa_repr, pair_repr):
        # MSA stack
        msa_repr, _ = self.msa_row_attn(msa_repr, pair_repr)
        
        # Column attention (transpose to attend over sequences)
        N_seq, L, c_m = msa_repr.shape
        msa_t = msa_repr.transpose(0, 1).reshape(L, N_seq, c_m)
        msa_col_out, _ = self.msa_col_attn(msa_t, msa_t, msa_t)
        msa_repr = msa_repr + msa_col_out.reshape(L, N_seq, c_m).transpose(0, 1)
        
        msa_repr = msa_repr + self.msa_transition(msa_repr)
        
        # Outer product mean: MSA -> Pair
        pair_repr = pair_repr + self.outer_product_mean(msa_repr)
        
        # Pair stack
        pair_repr = self.tri_mult_out(pair_repr)
        pair_repr = self.tri_mult_in(pair_repr)
        pair_repr, _ = self.tri_attn_start(pair_repr)
        pair_repr, _ = self.tri_attn_end(pair_repr)
        pair_repr = pair_repr + self.pair_transition(pair_repr)
        
        return msa_repr, pair_repr

# Test complete block
evoformer_block = EvoformerBlock(c_m, c_z)

msa_out, pair_out = evoformer_block(msa_repr, pair_repr)

print(f"Input MSA: {msa_repr.shape}, Pair: {pair_repr.shape}")
print(f"Output MSA: {msa_out.shape}, Pair: {pair_out.shape}")

In [ ]:
# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Evoformer block parameters: {count_parameters(evoformer_block):,}")

# AlphaFold2 uses 48 blocks
n_blocks = 48
print(f"Full Evoformer ({n_blocks} blocks): ~{count_parameters(evoformer_block) * n_blocks:,} parameters")

## 7. Stacked Evoformer

Let's see how the representations evolve through multiple blocks.

In [ ]:
class Evoformer(nn.Module):
    """Stacked Evoformer blocks."""
    
    def __init__(self, c_m=256, c_z=128, n_blocks=4):
        super().__init__()
        self.blocks = nn.ModuleList([
            EvoformerBlock(c_m, c_z) for _ in range(n_blocks)
        ])
        
    def forward(self, msa_repr, pair_repr, return_intermediates=False):
        intermediates = {'msa': [msa_repr], 'pair': [pair_repr]}
        
        for block in self.blocks:
            msa_repr, pair_repr = block(msa_repr, pair_repr)
            if return_intermediates:
                intermediates['msa'].append(msa_repr)
                intermediates['pair'].append(pair_repr)
        
        if return_intermediates:
            return msa_repr, pair_repr, intermediates
        return msa_repr, pair_repr

# Test with 4 blocks
evoformer = Evoformer(c_m, c_z, n_blocks=4)

with torch.no_grad():
    msa_final, pair_final, intermediates = evoformer(msa_repr, pair_repr, return_intermediates=True)

print(f"Number of intermediate states: {len(intermediates['pair'])}")

In [ ]:
# Visualize evolution of pair representation through blocks
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

channel = 0

for idx, (ax, pair_state) in enumerate(zip(axes.flat, intermediates['pair'])):
    if idx < len(intermediates['pair']):
        im = ax.imshow(pair_state[:, :, channel].detach().numpy(), cmap='RdBu_r', vmin=-3, vmax=3)
        ax.set_title(f'After Block {idx}' if idx > 0 else 'Initial')
        plt.colorbar(im, ax=ax)

plt.suptitle(f'Pair Representation Evolution (Channel {channel})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Track how pair representation changes
pair_changes = []
for i in range(1, len(intermediates['pair'])):
    diff = (intermediates['pair'][i] - intermediates['pair'][i-1]).abs().mean().item()
    pair_changes.append(diff)

plt.figure(figsize=(8, 4))
plt.bar(range(1, len(pair_changes)+1), pair_changes)
plt.xlabel('Block Number')
plt.ylabel('Mean Absolute Change')
plt.title('Pair Representation Change Per Block')
plt.show()

## 8. Key Takeaways

### What We Learned

1. **Dual Representation**: AlphaFold2 maintains MSA (evolutionary) and Pair (structural) representations that communicate bidirectionally.

2. **Triangular Updates**: The triangular multiplicative and attention operations enforce geometric consistency in the pair representation.

3. **Pair Bias**: MSA attention is biased by pair information, allowing structural knowledge to guide evolutionary signal processing.

4. **Outer Product Mean**: Co-evolutionary information from the MSA flows to the pair representation through outer products.

### Design Principles

| Component | Purpose |
|-----------|--------|
| MSA Row Attention | Process evolutionary patterns at each position |
| MSA Column Attention | Share information across homologous sequences |
| Triangular Multiply | Enforce geometric consistency via path closure |
| Triangular Attention | Learn complex pairwise interactions |
| Outer Product Mean | Transfer co-evolution signal to pair representation |

## 9. Exercises

1. **Modify the number of heads**: How does changing `n_heads` affect the attention patterns?

2. **Asymmetric pair representation**: Make the pair representation non-symmetric and observe how triangular updates behave.

3. **Dropout**: Add dropout to the Evoformer block. Where would it be most effective?

4. **Memory optimization**: Implement gradient checkpointing for the Evoformer to reduce memory usage.

In [ ]:
# Exercise: Experiment with different configurations

# Try different numbers of heads
for n_heads in [2, 4, 8, 16]:
    tri_attn = TriangularSelfAttention(c_z, n_heads=n_heads)
    _, attn_weights = tri_attn(pair_repr)
    print(f"n_heads={n_heads}: attention shape = {attn_weights.shape}")